In [ ]:
import cv2
import numpy as np

sift = cv2.SIFT_create()

def count_keypoints(imgs):
    counts = []
    for img in imgs:
        if img.dtype != np.uint8:
            img = img.astype(np.uint8)
        kp, desc = sift.detectAndCompute(img, None)
        counts.append(0 if kp is None else len(kp))
    return np.array(counts)

counts = count_keypoints(train_imgs)  # train_imgs from your Dataset

print("num images:", len(counts))
print("min keypoints:", counts.min())
print("max keypoints:", counts.max())
print("mean keypoints:", counts.mean())
print("median keypoints:", np.median(counts))
print("total keypoints:", counts.sum())


In [2]:
from dataset import Dataset

# N_NEIGHBORS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

# le = LabelEncoder()
training = Dataset(r"training",img_size=(256, 256))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_validate, labels_validate = imgs[split_index:], labels[split_index:]

testing = Dataset(r"testing",img_size=(256, 256), label_exist=False)
# testing.shuffle_data()
imgs_test, img_names_test = testing.get_data()

print("Number of testing images:", len(imgs_test))
print(img_names_test)


Number of testing images: 2985
['0.jpg' '1.jpg' '2.jpg' ... '2985.jpg' '2986.jpg' '2987.jpg']


In [3]:
from siftbovw import SiftBoVW
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

PARAM = {
    'n_words': 100,
    'max_sample': 0,
    'batch_size': 20000,
    'random_state': 0
}

N_NEIGHBORS = [5]
print("Parameters:", PARAM)

sift_bovw = SiftBoVW(n_words=PARAM['n_words'], max_sample=PARAM['max_sample'], batch_size=PARAM['batch_size'], random_state=PARAM['random_state'])
x_train = sift_bovw.fit_transform(imgs_train)

# Check the cluster centers
center = sift_bovw.kmeans.cluster_centers_

x_validate = sift_bovw.transform(imgs_validate)
x_test = sift_bovw.transform(imgs_test)

for N_NEIGHBOR in N_NEIGHBORS:
    print("Number of neighbors:", N_NEIGHBOR)
    knn = KNeighborsClassifier(n_neighbors= N_NEIGHBOR)
    knn.fit(x_train, labels_train)
    y_validate_pred = knn.predict(x_validate)
    y_test_pred = knn.predict(x_test)
    accuracy = accuracy_score(labels_validate, y_validate_pred)
    print(f"N_NEIGHBOR={N_NEIGHBOR}: Accuracy: {accuracy*100:.2f}%")
    for img_name, pred in zip(img_names_test, y_test_pred):
        print(f"Img name: {img_name}, Predicted label: {pred}")
    testing.runtxt_write(y_test_pred, filepath=f"runtxt_knn_{N_NEIGHBOR}.txt")
    

Parameters: {'n_words': 100, 'max_sample': 0, 'batch_size': 20000, 'random_state': 0}
[SIFTBoVW] Total descriptors before sampling: 696266
Number of neighbors: 5
N_NEIGHBOR=5: Accuracy: 44.67%
Img name: 0.jpg, Predicted label: Highway
Img name: 1.jpg, Predicted label: Forest
Img name: 2.jpg, Predicted label: Insidecity
Img name: 3.jpg, Predicted label: industrial
Img name: 4.jpg, Predicted label: Mountain
Img name: 5.jpg, Predicted label: livingroom
Img name: 6.jpg, Predicted label: Insidecity
Img name: 7.jpg, Predicted label: kitchen
Img name: 8.jpg, Predicted label: TallBuilding
Img name: 9.jpg, Predicted label: Street
Img name: 10.jpg, Predicted label: OpenCountry
Img name: 11.jpg, Predicted label: Street
Img name: 12.jpg, Predicted label: Street
Img name: 13.jpg, Predicted label: store
Img name: 14.jpg, Predicted label: Mountain
Img name: 15.jpg, Predicted label: Highway
Img name: 16.jpg, Predicted label: Forest
Img name: 17.jpg, Predicted label: store
Img name: 18.jpg, Predicted l

In [4]:
PARAM = {
    'n_words': 100,
    'max_sample': 0,
    'batch_size': 20000,
    'random_state': 0
}


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("=" * 60)
print("Non-linear SVM with Homogeneous Kernel Map")
print("=" * 60)

# Homogeneous Kernel Map + Linear SVM (optimized version)
print("\nAdditiveChi2Sampler + StandardScaler + LinearSVC:")
SAMPLE_STEPS = [2, 3]  # Reduce parameter combinations, focus on effective ones
C_VALUES = [0.01, 0.1, 1, 10, 100]  # Expand C value range

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        # Build optimized pipeline: Kernel Map -> Standardization -> Linear SVM
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('scaler', StandardScaler(with_mean=False)),  # No centering for sparse data
            ('svm', LinearSVC(C=C, max_iter=5000, dual='auto', random_state=PARAM['random_state']))
        ])
        pipeline.fit(x_train, labels_train)
        y_validate_pred = pipeline.predict(x_validate)
        accuracy = accuracy_score(labels_validate, y_validate_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")


Non-linear SVM with Homogeneous Kernel Map

AdditiveChi2Sampler + StandardScaler + LinearSVC:

  Sample steps: 2
    C=0.01: Accuracy: 46.67%
    C=0.1: Accuracy: 45.33%
    C=1: Accuracy: 37.33%
    C=10: Accuracy: 40.00%
    C=100: Accuracy: 39.33%

  Sample steps: 3
    C=0.01: Accuracy: 47.33%
    C=0.1: Accuracy: 44.00%
    C=1: Accuracy: 44.00%
    C=10: Accuracy: 44.00%
    C=100: Accuracy: 44.00%


In [ ]:
from phow import PHOW
from dataset import Dataset

from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# Load training and test images
training = Dataset(r"training",img_size=(256, 256))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_validate, labels_validate = imgs[split_index:], labels[split_index:]


print("=" * 60)
print("PHOW (Pyramid Histogram of Words) Features")
print("=" * 60)


# Configure parameters
PHOW_PARAM = {
    'n_words': 250,
    'max_sample': 200000,
    'batch_size': 100,
    'pyramid_levels': [0, 1, 2],  # 1x1, 2x2, 4x4 grids
    'random_state': 0
}

print(f"\nParameters: {PHOW_PARAM}")

# Train PHOW feature extractor
phow = PHOW(
    n_words=PHOW_PARAM['n_words'],
    max_sample=PHOW_PARAM['max_sample'],
    batch_size=PHOW_PARAM['batch_size'],
    pyramid_levels=PHOW_PARAM['pyramid_levels'],
    random_state=PHOW_PARAM['random_state']
)

# Extract features
print("\nExtracting training set features...")
x_train_phow = phow.fit_transform(imgs_train)
print(f"Training set feature shape: {x_train_phow.shape}")

print("\nExtracting test set features...")
x_validate_phow = phow.transform(imgs_validate)
print(f"Test set feature shape: {x_validate_phow.shape}")

# Use Homogeneous Kernel Map + LinearSVC
print("\n" + "=" * 60)
print("PHOW + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1, 10]

best_acc = 0
best_params = {}

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PHOW_PARAM['random_state']))
        ])
        pipeline.fit(x_train_phow, labels_train)
        y_validate_pred = pipeline.predict(x_validate_phow)
        accuracy = accuracy_score(labels_validate, y_validate_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")
        
        if accuracy > best_acc:
            best_acc = accuracy
            best_params = {'sample_steps': sample_steps, 'C': C}

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best parameters: {best_params}")


PHOW (Pyramid Histogram of Words) Features

Parameters: {'n_words': 250, 'max_sample': 200000, 'batch_size': 100, 'pyramid_levels': [0, 1, 2], 'random_state': 0}

Extracting training set features...
[PHOW] Extracting Dense SIFT descriptors...
[PHOW] Total descriptors: 1382400
[PHOW] Sampled 200000 descriptors
[PHOW] Training K-Means (n_clusters=250)...
[PHOW] Training complete! Feature dimension: 5250
[PHOW] Processed 100/1350 images
[PHOW] Processed 200/1350 images
[PHOW] Processed 300/1350 images
[PHOW] Processed 400/1350 images
[PHOW] Processed 500/1350 images
[PHOW] Processed 600/1350 images
[PHOW] Processed 700/1350 images
[PHOW] Processed 800/1350 images
[PHOW] Processed 900/1350 images
[PHOW] Processed 1000/1350 images
[PHOW] Processed 1100/1350 images
[PHOW] Processed 1200/1350 images
[PHOW] Processed 1300/1350 images
Training set feature shape: (1350, 5250)

Extracting test set features...
[PHOW] Processed 100/150 images
Test set feature shape: (150, 5250)

PHOW + Chi2 Kernel 

In [ ]:
from gist_feature import GISTFeature
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np


# Load training and test images
training = Dataset(r"training",img_size=(128, 128))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_validate, labels_validate = imgs[split_index:], labels[split_index:]


print("=" * 60)
print("GIST Global Scene Features")
print("=" * 60)

# Configure GIST parameters
gist = GISTFeature(n_scales=4, n_orientations=8, n_blocks=4)
print(f"GIST feature dimension: {gist.feature_dim}")

# Extract GIST features
print("\nExtracting training set GIST features...")
x_train_gist = gist.extract(imgs_train)
print(f"Training set GIST feature shape: {x_train_gist.shape}")

print("\nExtracting test set GIST features...")
x_validate_gist = gist.extract(imgs_validate)
print(f"Test set GIST feature shape: {x_validate_gist.shape}")

# Test different classifiers
print("\n" + "=" * 60)
print("GIST + Different Classifiers")
print("=" * 60)

# 1. Linear SVM
print("\n[1] GIST + LinearSVC:")
C_VALUES = [0.01, 0.1, 1, 10]
for C in C_VALUES:
    svm = LinearSVC(C=C, max_iter=2000, random_state=PARAM['random_state'])
    svm.fit(x_train_gist, labels_train)
    y_validate_pred = svm.predict(x_validate_gist)
    accuracy = accuracy_score(labels_validate, y_validate_pred)
    print(f"  C={C}: Accuracy: {accuracy*100:.2f}%")

# 2. Random Forest
print("\n[2] GIST + Random Forest:")
N_ESTIMATORS = [50, 100, 200]
for n_est in N_ESTIMATORS:
    rf = RandomForestClassifier(n_estimators=n_est, random_state=PARAM['random_state'], n_jobs=-1)
    rf.fit(x_train_gist, labels_train)
    y_validate_pred = rf.predict(x_validate_gist)
    accuracy = accuracy_score(labels_validate, y_validate_pred)
    print(f"  n_estimators={n_est}: Accuracy: {accuracy*100:.2f}%")


GIST Global Scene Features
GIST feature dimension: 512

Extracting training set GIST features...
[GIST] Processed 100/1350 images
[GIST] Processed 200/1350 images
[GIST] Processed 300/1350 images
[GIST] Processed 400/1350 images
[GIST] Processed 500/1350 images
[GIST] Processed 600/1350 images
[GIST] Processed 700/1350 images
[GIST] Processed 800/1350 images
[GIST] Processed 900/1350 images
[GIST] Processed 1000/1350 images
[GIST] Processed 1100/1350 images
[GIST] Processed 1200/1350 images
[GIST] Processed 1300/1350 images
Training set GIST feature shape: (1350, 512)

Extracting test set GIST features...
[GIST] Processed 100/150 images
Test set GIST feature shape: (150, 512)

GIST + Different Classifiers

[1] GIST + LinearSVC:
  C=0.01: Accuracy: 13.33%
  C=0.1: Accuracy: 16.67%
  C=1: Accuracy: 22.00%
  C=10: Accuracy: 32.67%

[2] GIST + Random Forest:
  n_estimators=50: Accuracy: 47.33%
  n_estimators=100: Accuracy: 51.33%
  n_estimators=200: Accuracy: 57.33%


In [ ]:
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import numpy as np

print("=" * 60)
print("Feature Fusion: PHOW + GIST")
print("=" * 60)

# Concatenate PHOW and GIST features
print("\nConcatenating features...")
x_train_combined = np.hstack([x_train_phow, x_train_gist])
x_validate_combined = np.hstack([x_validate_phow, x_validate_gist])

print(f"Combined training set feature shape: {x_train_combined.shape}")
print(f"Combined test set feature shape: {x_validate_combined.shape}")

# L2 normalization
x_train_combined = normalize(x_train_combined, norm='l2')
x_validate_combined = normalize(x_validate_combined, norm='l2')

# Use LinearSVC
print("\n" + "=" * 60)
print("Combined Features + LinearSVC")
print("=" * 60)

C_VALUES = [0.01, 0.1, 1, 10]
best_acc = 0
best_C = 0

for C in C_VALUES:
    svm = LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PARAM['random_state'])
    svm.fit(x_train_combined, labels_train)
    y_validate_pred = svm.predict(x_validate_combined)
    accuracy = accuracy_score(labels_validate, y_validate_pred)
    print(f"  C={C}: Accuracy: {accuracy*100:.2f}%")
    
    if accuracy > best_acc:
        best_acc = accuracy
        best_C = C

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best C value: {best_C}")

# Use Chi2 Kernel Map
print("\n" + "=" * 60)
print("Combined Features + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1]

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PARAM['random_state']))
        ])
        pipeline.fit(x_train_combined, labels_train)
        y_validate_pred = pipeline.predict(x_validate_combined)
        accuracy = accuracy_score(labels_validate, y_validate_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")


Feature Fusion: PHOW + GIST

Concatenating features...
Combined training set feature shape: (1350, 5762)
Combined test set feature shape: (150, 5762)

Combined Features + LinearSVC
  C=0.01: Accuracy: 54.00%
  C=0.1: Accuracy: 63.33%
  C=1: Accuracy: 66.67%
  C=10: Accuracy: 66.67%

Best accuracy: 66.67%
Best C value: 1

Combined Features + Chi2 Kernel Map + LinearSVC

  Sample steps: 2
    C=0.01: Accuracy: 66.67%
    C=0.1: Accuracy: 68.67%
    C=1: Accuracy: 67.33%
